# DiffusionSat speedup experimentation

The goal of this notebook is to improve the DiffusionSat backbone inference times.

### Experiments

Initial config:
- bs 32, num_workers 4, no explicit pytorch optims, 512 samples, max_ts 50, save_ts [48, 56, 42], layer_idxs down attn1 all

Baseline: 

In [1]:
%load_ext autoreload
%autoreload 2

import os
import warnings
from pathlib import Path

import torch
from tqdm import tqdm

# Filter warnings.
warnings.filterwarnings("ignore", message=".*invalid escape sequence.*")


VISLOC_ROOT = Path(os.environ["VISLOC_ROOT"])
DIFFUSIONSAT_256_CHCKPT = Path(os.environ["DIFFUSIONSAT_256_CHCKPT"])

RANDOM_SEED = 42
DEVICE = torch.device("cuda")
NUM_WORKERS = 4

In [ ]:
from src.backbone import DiffusionSatBackbone
from src.ldm_extractor import LDMExtractorCfg

# Torch performance.
# torch.set_float32_matmul_precision("high")

# Initialize the backbone once to avoid re-loading unet to VRAM each time.
diffusionsat_backbone = DiffusionSatBackbone(DIFFUSIONSAT_256_CHCKPT, DEVICE)

ldm_extractor_cfg = LDMExtractorCfg(
  batch_size=32,
  save_timesteps=[48, 46, 42],
  num_timesteps=50,
  layer_idxs={"down_blocks": {"attn1": "all"}},
)
diffusionsat_backbone.set_ldm_extractor_cfg(ldm_extractor_cfg)

/root/diffusion-vpr/.venv/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/root/diffusion-vpr/.venv/lib/python3.10/site-packages/diffusers/models/cross_attention.py:30: FutureWarning: Importing from cross_attention is deprecated. Please import from diffusers.models.attention_processor instead.
  deprecate(


In [ ]:
from torch.utils.data import Subset

from src.data_transforms import inference_transforms
from src.datasets import SatChunkDataset

gallery_dataset = SatChunkDataset(
  root=VISLOC_ROOT,
  flight_id="03",
  transform=inference_transforms,
)
gallery_dataset = Subset(gallery_dataset, range(512))
loader = torch.utils.data.DataLoader(
  gallery_dataset,
  batch_size=ldm_extractor_cfg.batch_size,
  shuffle=False,
  num_workers=NUM_WORKERS,
  pin_memory=True,
)


In [ ]:
%%timeit
for batch in tqdm(loader, desc="Extracting features"):
  images = batch["image"].to(DEVICE)
  features = diffusionsat_backbone(images)